### import packages

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import dgl
import random
from scipy.sparse import csr_matrix
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score

C:\Users\10360\anaconda3\envs\CAMEX_t\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from stAggregator import stAggregator
from stAggregator import metrics
from stAggregator.data import process_adata, process_graph, mclust_R_smooth
from stAggregator.data import process_graph

In [5]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
dgl.random.seed(seed)

In [6]:
def refine_label(adata, radius=50, key='label'):
    import ot
    n_neigh = radius
    new_type = []
    old_type = adata.obs[key].values

    # calculate distance
    position = adata.obsm['spatial']
    distance = ot.dist(position, position, metric='euclidean')

    n_cell = distance.shape[0]

    for i in range(n_cell):
        vec = distance[i, :]
        index = vec.argsort()
        neigh_type = []
        for j in range(1, n_neigh + 1):
            neigh_type.append(old_type[index[j]])
        max_type = max(neigh_type, key=neigh_type.count)
        new_type.append(max_type)

    new_type = [str(i) for i in list(new_type)]

    return new_type

In [7]:
def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=42):
    """\
    Clustering using the mclust algorithm.
    The parameters are the same as those in the R package mclust.
    """

    np.random.seed(random_seed)
    import rpy2.robjects as robjects
    robjects.r.library("mclust")

    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
    r_random_seed = robjects.r['set.seed']
    r_random_seed(random_seed)
    rmclust = robjects.r['Mclust']

    res = rmclust(rpy2.robjects.numpy2ri.numpy2rpy(adata.obsm[used_obsm]), num_cluster, modelNames)
    mclust_res = np.array(res[-2])

    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int')
    adata.obs['mclust'] = adata.obs['mclust'].astype('category')
    adata.obsm['mclust_prob'] = np.array(res[-3])
    return adata

In [8]:
def mclust_R_smooth(adata, num_cluster, modelNames='EEE', used_obsm='X_pca', radius=50, random_seed=42):
    """\
    Clustering using the mclust algorithm.
    The parameters are the same as those in the R package mclust.
    """

    np.random.seed(random_seed)
    import rpy2.robjects as robjects
    robjects.r.library("mclust")

    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
    r_random_seed = robjects.r['set.seed']
    r_random_seed(random_seed)
    rmclust = robjects.r['Mclust']

    res = rmclust(rpy2.robjects.numpy2ri.numpy2rpy(adata.obsm[used_obsm]), num_cluster, modelNames)
    mclust_res = np.array(res[-2])

    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int')
    adata.obs['mclust'] = adata.obs['mclust'].astype('category')
    adata.obsm['mclust_prob'] = np.array(res[-3])

    adata.obs['mclust'] = refine_label(adata, radius=radius, key='mclust')

    return adata

In [9]:
from stAggregator.metrics import evaluate_all

In [11]:
sc.set_figure_params(dpi=150, figsize=(2, 2), frameon=False)    

In [12]:
def clear_fig(fig):
    if fig:
        fig.axes[0].set_xlabel(None)
        fig.axes[0].set_ylabel(None)
        fig.tight_layout()
    else:
        pass
    return fig

### load data and train

In [15]:
ARI_list = []
NMI_list = []
metrics_list = []
Batch_list = []
adj_list = []
# section_ids = ['151673','151674','151675','151676']
section_ids = [
                 '151507',
                 '151508',
                 '151509',
                 '151510',

                 '151673',
                 '151674',
                 '151675',
                 '151676',

                 '151669',
                 '151670',
                 '151671',
                 '151672',
]
cluster_num = [7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5]

for i, section_id in enumerate(section_ids):
    print(section_id)
    adata1 = sc.read_h5ad(f'./data/0DLPFC/{section_id}.h5ad')
    adata1.obs_names_make_unique()
    adata1.var_names_make_unique()
    data_list = [adata1]
    batch_key = 'batch'
    batch_names = ['adata1']

    adata = process_adata(data_list, batch_key=batch_key, batch_categories=batch_names, n_top_features=3000)
    rad_cutoff_list = [150]
    adata = process_graph(adata, data_list, rad_cutoff_list=rad_cutoff_list)
    path_results = f'./log/integration/'
    adata_stAggregator = stAggregator(adata=adata, max_iteration=800, outdir=path_results, h_dim=16,
                           batch_size=19999, impute=False, early_stop=False, random_seed=42)

    
    # n_clusters=7
    adata_stAggregator = mclust_R_smooth(adata_stAggregator, used_obsm='X_stAggregator', num_cluster=cluster_num[i])
    adata_stAggregator.obs['mclust'] = adata_stAggregator.obs['mclust']
    # stagate
    sc.pp.neighbors(adata_stAggregator, use_rep='X_stAggregator')
    sc.tl.umap(adata_stAggregator)
    # clear_fig(sc.pl.umap(adata_stAggregator, color='mclust', title='', legend_loc=None, return_fig=True)).savefig(f'./umap/{section_id}.jpg')
    # clear_fig(sc.pl.embedding(adata_stAggregator, color='mclust', title='', legend_loc=None, basis='spatial', return_fig=True)).savefig(f'./spatial/{section_id}.jpg')
    
    adata_new = adata_stAggregator[~adata_stAggregator.obs.loc[:, 'Region'].isnull(), :]
    ari = ari_score(adata_new.obs['Region'], adata_new.obs['mclust'])
    nmi = normalized_mutual_info_score(adata_new.obs['Region'], adata_new.obs['mclust'])
    print('mclust, ARI = %01.3f' % ari)
    print('mclust, NMI = %01.3f' % nmi)
    ARI_list.append(ari)
    NMI_list.append(nmi)
    
    # adata_raw = sc.read_h5ad(f'./data/{section_id}.h5ad')
    # adata_raw = adata_raw[adata_new.obs.index, :].copy()
    # adata_raw.obs.loc[:, 'mclust'] = adata_new.obs.loc[:, 'mclust']
    # metrics = evaluate_all(adata_raw, gt_key='Region', pred_key='mclust')
    # metrics_list.append(metrics)
    # break

151507


E:\学习\6科研\论文\博士论文\stAggregator\stAggregator\stAggregator\data.py:87: ImplicitModificationWarning: Setting element `.obsm['feature_scale']` of view, initializing view as actual.
  adata.obsm['feature_scale'] = adata_scale.X


------Calculating spatial graph...


E:\学习\6科研\论文\博士论文\stAggregator\stAggregator\stAggregator\data.py:142: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Spatial_Net['Cell1'] = Spatial_Net['Cell1'].map(id_cell_trans)
E:\学习\6科研\论文\博士论文\stAggregator\stAggregator\stAggregator\data.py:143: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Spatial_Net['Cell2'] = Spatial_Net['Cell2'].map(id_cell_trans)


The graph contains 24770 edges, 4226 cells.
5.8613 neighbors per cell on average.


...

------Calculating spatial graph...


E:\学习\6科研\论文\博士论文\stAggregator\stAggregator\stAggregator\data.py:142: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Spatial_Net['Cell1'] = Spatial_Net['Cell1'].map(id_cell_trans)
E:\学习\6科研\论文\博士论文\stAggregator\stAggregator\stAggregator\data.py:143: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Spatial_Net['Cell2'] = Spatial_Net['Cell2'].map(id_cell_trans)


The graph contains 23382 edges, 4015 cells.
5.8237 neighbors per cell on average.


Epochs: 100%|████████████████████████████████████████████████████████████████████| 400/400 [00:14<00:00, 26.97it/s, recon_loss_mse=467.815, sl1_loss=1.375, sl2_loss=1.375, lr: 0.001]
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INFO - Output dir: ./log/integration/
2025-09-28 13:38:21,814 - root - INF

fitting ...
  |======================================================================| 100%
mclust, ARI = 0.578
mclust, NMI = 0.703


In [16]:
np.mean(ARI_list), np.mean(NMI_list)

(0.5751141791723521, 0.678769442407175)